# 02: Valuing every touch

VAEP in one sentence: an action is worth how much it changes your team's chance of scoring, minus how much it changes the opponent's. A through ball that creates a 30% chance is worth about +0.25. A sideways pass at halfway is worth about nothing. A goal is worth 1 minus the chance it came from.

In [1]:
from effc.vaep.features import featurize_match, K
from effc.data.statsbomb import load_events, list_matches
from effc.data.download import TARGET

mid = int(list_matches([TARGET])['match_id'].iloc[0])
df = featurize_match(load_events(mid), match_id=mid)
print(f"{len(df)} actions, K={K}")
print(df[['player','action','start_x','dist_start','scores_incl','scores_excl']].head(5).to_string(index=False))

2318 actions, K=10
              player action  start_x  dist_start  scores_incl  scores_excl
   Ikoma Loïs Openda   Pass     61.0   44.420828          0.0          0.0
Daniel Olmo Carvajal   Pass     56.2   49.271493          0.0          0.0
   Benjamin Henrichs  Carry     55.0   64.156371          0.0          0.0
   Benjamin Henrichs   Pass     56.4   63.199051          0.0          0.0
     Mohamed Simakan  Carry     37.8   74.646433          0.0          0.0


Two label windows, and the asymmetry is the whole trick. The *including* window counts the action's own goal, so a shot's danger is visible before it. The *excluding* window does not, so the state after a goal is not poisoned by the goal that just happened. Getting this wrong gives goals zero value or keepers infinite value. I got it wrong twice before landing here.

In [2]:
import pickle, os
from effc.data.download import DATA_DIR
from effc.vaep.models import train_goal_models, brier_score, predict_proba

frames = pickle.load(open(os.path.join(DATA_DIR, 'cache', 'pool_frames.pkl'), 'rb'))
models = train_goal_models(frames, seed=0)
import pandas as pd
full = pd.concat(frames, ignore_index=True)
for lbl in ['scores_incl', 'concedes_incl']:
    p = predict_proba(models, full)[lbl]
    print(lbl, 'Brier:', round(brier_score(full[lbl].to_numpy(), p), 4))

scores_incl Brier: 0.0085


concedes_incl Brier: 0.0021


Brier scores around 0.008. For rare events that is well calibrated. Now value the match: watch what a goal, an assist, and a save are worth.

In [3]:
from effc.vaep.values import add_vaep

df = add_vaep(df, models)
goals = df[df['is_goal']]
print('GOALS:')
print(goals[['player','action','vaep','vaep_off']].to_string(index=False))
print()
saves = df[(df['action']=='Goal Keeper')].nlargest(3, 'vaep_def')
print('TOP SAVES (defensive value):')
print(saves[['player','vaep','vaep_def']].to_string(index=False))
print()
print('mean VAEP by action:')
print(df.groupby('action')['vaep'].mean().sort_values(ascending=False).round(4).head(8).to_string())

GOALS:
              player action     vaep  vaep_off
    Jeremie Frimpong   Shot 0.831664  0.832841
        Jonathan Tah   Shot 0.961362  0.962867
Daniel Olmo Carvajal   Shot 0.941774  0.941594
       Florian Wirtz   Shot 0.939382  0.940338
   Ikoma Loïs Openda   Shot 0.956548  0.958026

TOP SAVES (defensive value):
        player     vaep  vaep_def
Lukáš Hrádecký 0.501086  0.497217
Janis Blaswich 0.067367  0.067085
Lukáš Hrádecký 0.055416  0.053087

mean VAEP by action:
action
Shot             0.1707
Ball Recovery    0.0132
Interception     0.0118
Clearance        0.0094
Block            0.0043
Pass             0.0031
Carry            0.0027
Duel             0.0022


Goals land around +0.7 (one goal minus the chance it came from). Saves land around +0.25. The keeper who concedes gets -0.7ish via the conceded-goal debit, which is what keeps keeper ratings honest. This all matches the theory, which is reassuring because the theory is the only thing I had to go on.